# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 3: *Feature Engineering*
##### Version Number: 3.0
---
### Contents  
> 1. *Add Mean income per county*
> 2. *Calculate Temporal Fields*
> 3. *Calculate Lagged Weather Variables*
> 4. *Build Target*
> 5. *Fire History*
> 6. *File Export*
---
### Notes
> Features Added:
> - `Days_without_Rain` A rolling count that serves as a simple drought indicator
> - `Fire_History` An average count of fires per month in a region spanning the alst two years (needs refinement)
> - `Lagged_Variables` 7 day rolling averages for 'Avg Air Temp (F)', 'Precip (in)', 'Avg Rel Hum (%)', 'Avg Wind Speed (mph)'
---
### Inputs
- `trimmed.csv` cleaned weather data joined with cleaned fire damage dataset
---
### Outputs 
- `final.csv` - final clean dataset with features added
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Show basic facts about a dataframe
from src.data_utils import basic_explore

# basic health checks after a merge
from src.data_utils import post_merge_check

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

---

## Load Datasets

In [3]:
samples = pd.read_csv("../data/processed/samples_projected.csv")
mean_income = pd.read_csv("../data/raw/mean_income_by_county.csv")

---

## 2. Calculate Temporal Fields
`Winter` = 0, `Spring` = 1, `Summer` = 2, `Fall` = 3

In [4]:
def get_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return 0
    elif month in [3, 4, 5]:
        return 1
    elif month in [6, 7, 8]:
        return 2
    else:
        return 3
    
## Apply function   
samples['Date'] = pd.to_datetime(samples['Date'])
samples['Season'] = samples['Date'].apply(get_season)
samples['Month'] = samples['Date'].dt.month
samples['Year'] = samples['Date'].dt.year

## 3. Calculate Lagged Weather Variables

We calculate 7-day rolling averages for select weather variables to capture recent trends that may influence fire severity.

In [5]:
# Sort data by County and Date to prepare for rolling average
samples = samples.sort_values(by=['grid_id', 'Date'])

# Define columns for 7-day rolling average
avg_columns = [
    "Vapor Pressure Deficit",
    "Precipitation",
    "Solar Radiation",
    "Daily Maximum Air Temperature",
    "Daily Minimum Air Temperature",
    "Maximum Relative Humidity",
    "Minimum Relative Humidity",
    "Wind Speed"
]

# Apply rolling mean by County
for col in avg_columns:
    samples[f'{col} 7 Day Avg'] = samples.groupby('grid_id')[col].rolling(window=7, min_periods=1).mean().reset_index(level=0, drop=True)

## 4. Build Final Target

Create a categorical target that represents the risk levels of damage caused.\
\
`Low` Risk = 0, No fires started on the day\
`Medium` Risk = 1, Fires present but limited property damage done, under 1 million dollars total\
`High` Risk = 2, Fires causing over 1 million dollars in damage 

In [6]:
def fire_risk_category(row):
    # High risk: damaging fires
    if (row['total_fire_damage'] > 0) or (row['acres'] > 0):
        return 2

    # Moderate risk: at least one fire
    elif row['fire_count'] > 0:
        return 1

    # Low risk: all other cases
    else:
        return 0

## Apply function
samples['Target'] = samples.apply(fire_risk_category, axis=1)

## Display resulting risk category assignments
samples['Target'].value_counts()

Target
0    506537
2     56789
1     45554
Name: count, dtype: int64

## 5.  Historical fires per month

Create a new column that records the historical average number of fires for each month, based on all previous years up to that point. Each row should reflect the average number of fires that occurred in the same calendar month across prior years, providing a historical baseline for comparison.

**ASSUMPTION** -Since the weather coverage in this dataset is limited, no previous year readings exists.
For now, I will just use the current count as a rough estimation for the average. 

In [7]:
# Convert Datetime column to string
samples['Month_Year'] = samples['Date'].dt.strftime('%Y-%m')
month_year = samples['Month_Year'].unique()

In [8]:
# Dictionary to store month-to-count mapping
monthly_fire_counts = {}

# Step 1: Calculate fire counts for each month
for month in month_year:
    # Get index and subset
    month_index = samples[samples['Month_Year'] == month].index
    subset = samples.loc[month_index]
    
    # Eliminate 'Low' days
    total = subset[subset['Target'] != 0]
    
    # Store count
    monthly_fire_counts[month] = len(total)

# Step 2: Calculate 2-year rolling average
# Convert month_year to sorted list to ensure order
month_year_sorted = sorted(month_year)

# Dictionary for rolling averages
rolling_avg = {}

for i, month in enumerate(month_year_sorted):
    if i == 0:
        # First year: average of all *future* years (exclude current)
        future_values = list(monthly_fire_counts.values())[1:]
        rolling_avg[month] = np.mean(future_values) if future_values else 0

    elif i == 1:
        # Second year: average of first year and current
        prev = month_year_sorted[0]
        rolling_avg[month] = np.mean([monthly_fire_counts[prev], monthly_fire_counts[month]])
    
    else:
        # Rolling average of previous 2 years
        prev1 = month_year_sorted[i - 1]
        prev2 = month_year_sorted[i - 2]
        rolling_avg[month] = np.mean([monthly_fire_counts[prev1], monthly_fire_counts[prev2]])

# Step 3: Assign back to each row in the dataframe
for month in month_year_sorted:
    month_index = samples[samples['Month_Year'] == month].index
    samples.loc[month_index, '2-Year Avg Fires'] = rolling_avg[month]

In [9]:
drop = ['Month_Year']
samples = samples.drop(columns=drop)

In [10]:
samples

,Sample_ID,Date,Burning Index,Energy Release Component,Actual Evapotranspiration,100-hour Dead Fuel Moisture,1000-hour Dead Fuel Moisture,Precipitation,Maximum Relative Humidity,Minimum Relative Humidity,...,Vapor Pressure Deficit 7 Day Avg,Precipitation 7 Day Avg,Solar Radiation 7 Day Avg,Daily Maximum Air Temperature 7 Day Avg,Daily Minimum Air Temperature 7 Day Avg,Maximum Relative Humidity 7 Day Avg,Minimum Relative Humidity 7 Day Avg,Wind Speed 7 Day Avg,Target,2-Year Avg Fires
0,1,2018-01-01,24.0,37.0,1.7,16.600000,13.700000,0.0,97.800003,52.700001,...,0.410000,0.000000,145.600006,290.700012,283.000000,97.800003,52.700001,1.300000,0,1215.119048
438,1,2018-01-02,25.0,39.0,2.0,15.900000,13.800000,0.0,90.599998,43.200001,...,0.545000,0.000000,138.900002,292.550003,284.149994,94.200001,47.950001,1.250000,0,1215.119048
707,1,2018-01-03,23.0,36.0,1.6,15.900000,13.800000,0.0,93.500000,53.400002,...,0.530000,0.000000,115.900002,292.666667,284.766663,93.966667,49.766668,1.233333,0,1215.119048
919,1,2018-01-04,27.0,38.0,2.3,16.299999,14.100000,0.0,98.300003,47.700001,...,0.537500,0.000000,124.725000,293.050003,284.824997,95.050001,49.250001,1.400000,0,1215.119048
1027,1,2018-01-05,31.0,38.0,3.1,16.000000,14.200000,0.0,89.699997,46.299999,...,0.564000,0.000000,130.100002,293.360004,285.159998,93.980000,48.660001,1.720000,0,1215.119048
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
607851,172,2025-01-19,0.0,8.0,1.4,19.299999,23.700001,1.0,100.000000,34.500000,...,0.314286,0.142857,115.514287,280.357143,265.342861,95.114286,36.900000,2.257143,0,1094.500000
608066,172,2025-01-20,19.0,16.0,1.6,19.000000,23.400000,0.0,95.099998,36.299999,...,0.321429,0.142857,117.014287,280.271428,265.085715,94.414286,35.442857,2.471429,0,1094.500000
608339,172,2025-01-21,18.0,19.0,2.0,17.600000,23.000000,0.0,86.500000,24.000000,...,0.327143,0.142857,118.757143,280.199999,264.842856,92.757143,34.514285,2.485714,0,1094.500000
608474,172,2025-01-22,20.0,21.0,2.4,16.000000,22.700001,0.0,79.000000,22.799999,...,0.321429,0.142857,120.542858,279.914285,264.214281,92.714286,33.714285,2.371429,0,1094.500000


---

## 6. Export Data

In [11]:
samples.to_csv("../data/processed/samples_income.csv", index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
